In [1]:
import torch
from transformers import AutoProcessor, AutoModel
from PIL import Image
import os
import pandas as pd
from tqdm import tqdm

# Configuration 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_WEIGHTS_PATH = "siglip_head_best.pth" 

IMAGES_DIR = "sim_cross_view_dataset/images2"
EXCEL_FILE_PATH = os.path.join("sim_cross_view_dataset", "image_data.xlsx")

# Load Teacher Model 
def load_teacher_model(weights_path):
    print(f"Loading teacher model on device: {DEVICE}")
    processor = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
    model = AutoModel.from_pretrained("google/siglip-base-patch16-224").to(DEVICE)
    model.eval()
    
    hidden_size = model.config.vision_config.hidden_size 
    head = torch.nn.Linear(hidden_size, 1).to(DEVICE)
    
    if not os.path.exists(weights_path):
        print(f"Error: Model weights not found at {weights_path}")
        return None, None, None
        
    head.load_state_dict(torch.load(weights_path, map_location=DEVICE))
    head.eval()
    
    print("Teacher model and trained head loaded successfully.")
    return model, head, processor

@torch.no_grad()
def get_aesthetic_score(image_pil, model, head, processor):
    inputs = processor(images=image_pil, return_tensors="pt")['pixel_values'].to(DEVICE)
    features = model.get_image_features(pixel_values=inputs)
    score = head(features).squeeze(-1).item()
    return score
'''
# Main Scoring Loop
def score_all_images(model, head, processor):
    print(f"Finding images in: {IMAGES_DIR}")
    
    # Find all the images we just rendered
    image_files = [f for f in os.listdir(IMAGES_DIR) if f.endswith(".png")]
    print(f"Found {len(image_files)} images to score.")
    
    dataset_records = []
    
    for img_filename in tqdm(image_files, desc="Scoring Images"):
        img_path = os.path.join(IMAGES_DIR, img_filename)
        img_path_relative = os.path.join(os.path.basename(IMAGES_DIR), img_filename)

        try:
            # Open the image
            frame_img = Image.open(img_path).convert("RGB")
            
            # Get the score
            aesthetic_score = get_aesthetic_score(frame_img, model, head, processor)
            
            # Save the record
            dataset_records.append((img_path_relative, aesthetic_score))
            
        except Exception as e:
            print(f"Error scoring {img_filename}: {e}")

    # Save to CSV 
    print(f"Finished scoring {len(dataset_records)} frames.")
    
    df = pd.DataFrame(dataset_records, columns=["image_path", "score"])
    
    # Sort by score to make it easy for Yixuan to find good/bad pairs
    df = df.sort_values(by="score", ascending=False)
    
    df.to_csv(CSV_FILE_PATH, index=False)
    print(f"Saved dataset labels to {CSV_FILE_PATH}")

if __name__ == "__main__":
    teacher_model, teacher_head, teacher_processor = load_teacher_model(MODEL_WEIGHTS_PATH)
    if teacher_model:
        score_all_images(teacher_model, teacher_head, teacher_processor)
    else:
        print("Could not load model. Aborting.")
'''

'\n# Main Scoring Loop\ndef score_all_images(model, head, processor):\n    print(f"Finding images in: {IMAGES_DIR}")\n    \n    # Find all the images we just rendered\n    image_files = [f for f in os.listdir(IMAGES_DIR) if f.endswith(".png")]\n    print(f"Found {len(image_files)} images to score.")\n    \n    dataset_records = []\n    \n    for img_filename in tqdm(image_files, desc="Scoring Images"):\n        img_path = os.path.join(IMAGES_DIR, img_filename)\n        img_path_relative = os.path.join(os.path.basename(IMAGES_DIR), img_filename)\n\n        try:\n            # Open the image\n            frame_img = Image.open(img_path).convert("RGB")\n            \n            # Get the score\n            aesthetic_score = get_aesthetic_score(frame_img, model, head, processor)\n            \n            # Save the record\n            dataset_records.append((img_path_relative, aesthetic_score))\n            \n        except Exception as e:\n            print(f"Error scoring {img_filename

In [ ]:
def score_dataset_from_excel(model, head, processor):
    print(f"Loading metadata from: {EXCEL_FILE_PATH}")
    
    if not os.path.exists(EXCEL_FILE_PATH):
        print("Error: Excel file not found. Please run generate_images.ipynb first.")
        return

    # Read the existing Excel file
    df = pd.read_excel(EXCEL_FILE_PATH)
    print(f"Found {len(df)} entries in the Excel sheet.")
    
    # Iterate through the DataFrame rows
    for index, row in tqdm(df.iterrows(), total=len(df), desc="Scoring Images"):
        img_filename = row['Filename']
        
        # Construct full path to the image
        img_path = os.path.join(IMAGES_DIR, img_filename)
        
        if not os.path.exists(img_path):
            print(f"Warning: Image not found at {img_path}")
            continue

        try:
            frame_img = Image.open(img_path).convert("RGB")
            
            score = get_aesthetic_score(frame_img, model, head, processor)
            
            df.at[index, 'Score'] = score
            
        except Exception as e:
            print(f"Error scoring {img_filename}: {e}")
            df.at[index, 'Score'] = 0.0 # Handle errors 

    # Save the updated DataFrame back to Excel
    print("Saving updated scores to Excel...")
    df.to_excel(EXCEL_FILE_PATH, index=False)
    print(f"Success! Updated dataset saved to {EXCEL_FILE_PATH}")

In [3]:
if __name__ == "__main__":
    teacher_model, teacher_head, teacher_processor = load_teacher_model(MODEL_WEIGHTS_PATH)
    
    if teacher_model:
        score_dataset_from_excel(teacher_model, teacher_head, teacher_processor)
    else:
        print("Could not load model. Aborting.")

Loading teacher model on device: cpu
Teacher model and trained head loaded successfully.
Loading metadata from: sim_cross_view_dataset/image_data.xlsx
Found 519 entries in the Excel sheet.


Scoring Images: 100%|█████████████████████████| 519/519 [00:37<00:00, 14.00it/s]

Saving updated scores to Excel...
Success! Updated dataset saved to sim_cross_view_dataset/image_data.xlsx
